In [1]:
{
  "nbformat": 4,
  "nbformat_minor": 0,
  "metadata": {
    "colab": {
      "provenance": []
    },
    "kernelspec": {
      "name": "python3",
      "display_name": "Python 3"
    },
    "language_info": {
      "name": "python"
    }
  },
  "cells": [
    {
      "cell_type": "code",
      "execution_count": 21,
      "metadata": {
        "id": "wJr6c-h6LXMR"
      },
      "outputs": [],
      "source": [
        "!pip install groq -q"
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "import os\n",
        "from google.colab import userdata\n",
        "from groq import Groq\n",
        "\n",
        "# Retrieve Groq API key from Colab secrets\n",
        "try:\n",
        "    GROQ_API_KEY = userdata.get('GROQ_API_KEY')\n",
        "except Exception:\n",
        "    print(\"Please add 'GROQ_API_KEY' to your Colab secrets (the key icon on the left) before continuing.\")\n",
        "    GROQ_API_KEY = None\n",
        "\n",
        "if GROQ_API_KEY:\n",
        "    client = Groq(api_key=GROQ_API_KEY)\n",
        "    print(\"Groq client successfully initialized!\")"
      ],
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "id": "WA-ErAA7LiY_",
        "outputId": "07a01ac8-d624-4ff5-8970-ce9d315e5b23"
      },
      "execution_count": 22,
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            "Groq client successfully initialized!\n"
          ]
        }
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "class NutritionAgent:\n",
        "    def __init__(self, client, model=\"openai/gpt-oss-120b\"):\n",
        "        self.client = client\n",
        "        self.model = model\n",
        "        # Initialize conversation memory with a system prompt\n",
        "        self.system_prompt = \"Act as a professional nutrition and meal analysis expert.\"\n",
        "        self.messages = [\n",
        "            {\"role\": \"system\", \"content\": self.system_prompt}\n",
        "        ]\n",
        "\n",
        "    def analyze_meal(self, meal_description):\n",
        "        # Append the new user message to the memory\n",
        "        self.messages.append({\"role\": \"user\", \"content\": meal_description})\n",
        "\n",
        "        try:\n",
        "            completion = self.client.chat.completions.create(\n",
        "                model=self.model,\n",
        "                messages=self.messages,\n",
        "                temperature=0.2\n",
        "            )\n",
        "            # Extract response\n",
        "            response = completion.choices[0].message.content\n",
        "\n",
        "            # Save the agent's response into memory\n",
        "            self.messages.append({\"role\": \"assistant\", \"content\": response})\n",
        "            return response\n",
        "        except Exception as e:\n",
        "            return f\"An error occurred: {e}\"\n",
        "\n",
        "    def clear_memory(self):\n",
        "        \"\"\"Resets the conversation memory.\"\"\"\n",
        "        self.messages = [\n",
        "            {\"role\": \"system\", \"content\": self.system_prompt}\n",
        "        ]\n",
        "        print(\"Conversation memory cleared.\")\n",
        "\n",
        "# Instantiate the agent securely checking if client is defined\n",
        "if 'client' in globals():\n",
        "    agent = NutritionAgent(client=client)\n",
        "    print(\"Nutrition Agent with memory is ready!\")\n",
        "else:\n",
        "    agent = None"
      ],
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "id": "j7bAgVHgLp1F",
        "outputId": "0ab3658d-c1bd-4a25-c239-b3a4c59e74b4"
      },
      "execution_count": 23,
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            "Nutrition Agent with memory is ready!\n"
          ]
        }
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "# We can test the memory-enabled nutrition agent here.\n",
        "# First introduction\n",
        "sample_meal = \"Hi I'm Chandru, and I'm planning to eat oatmeal with bananas for breakfast.\"\n",
        "\n",
        "if agent is not None:\n",
        "    print(\"--- Turn 1 ---\")\n",
        "    analysis1 = agent.analyze_meal(sample_meal)\n",
        "    print(f\"User: {sample_meal}\\n\")\n",
        "    print(f\"Agent: {analysis1}\\n\")\n",
        "\n",
        "    # Follow-up turn to verify if it remembers the user's name or previous meal\n",
        "    follow_up = \"Can you tell me what fruits I added to my breakfast, and mention my name?\"\n",
        "    print(\"--- Turn 2 (Testing Memory) ---\")\n",
        "    analysis2 = agent.analyze_meal(follow_up)\n",
        "    print(f\"User: {follow_up}\\n\")\n",
        "    print(f\"Agent: {analysis2}\\n\")\n",
        "else:\n",
        "    print(\"Cannot test. The Nutrition Agent has not been successfully initialized because the Groq client is missing.\")"
      ],
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "id": "_d_8lGL9L8A1",
        "outputId": "5167dc00-58ab-4dbe-e3eb-299c0abf494e"
      },
      "execution_count": 24,
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            "--- Turn 1 ---\n",
            "User: Hi I'm Chandru, and I'm planning to eat oatmeal with bananas for breakfast.\n",
            "\n",
            "Agent: **Hello Chandru!** 👋  \n",
            "It’s great that you’re starting the day with oatmeal and bananas – a classic combo that’s both tasty and nutrient‑dense. Below is a quick nutritional snapshot, the key health benefits, and a few ideas to make the meal even more balanced for different goals.\n",
            "\n",
            "---\n",
            "\n",
            "## 1️⃣ Basic Nutrition Profile (per typical serving)\n",
            "\n",
            "| Component | Approx. Amount* |\n",
            "|-----------|-----------------|\n",
            "| **Calories** | 300–350 kcal |\n",
            "| **Carbohydrates** | 55–65 g (≈ 70 % of calories) |\n",
            "| **Fiber** | 6–9 g (soluble & insoluble) |\n",
            "| **Protein** | 7–10 g |\n",
            "| **Fat** | 4–6 g (mostly from oats) |\n",
            "| **Sugar (natural)** | 12–15 g (mostly from banana) |\n",
            "| **Key Micronutrients** | Potassium (≈ 400 mg), Magnesium, Phosphorus, Vitamin B6, Iron, Folate, Vitamin C (from banana) |\n",
            "\n",
            "\\*Values are for **½ cup dry rolled oats** cooked with water + **½ – 1 medium banana**. Adjust portions to fit your needs.\n",
            "\n",
            "---\n",
            "\n",
            "## 2️⃣ Why This Pair Works Well\n",
            "\n",
            "| Benefit | How It Happens |\n",
            "|---------|----------------|\n",
            "| **Sustained Energy** | Oats provide complex carbs and β‑glucan fiber that slow glucose absorption, while bananas add quick‑acting carbs for an early‑morning boost. |\n",
            "| **Heart Health** | β‑glucan can lower LDL‑cholesterol; potassium from banana helps regulate blood pressure. |\n",
            "| **Digestive Support** | Soluble fiber (oats) feeds beneficial gut bacteria; insoluble fiber (banana skin remnants) adds bulk. |\n",
            "| **Mood & Brain** | B‑vitamins (especially B6) and magnesium support neurotransmitter synthesis and stress resilience. |\n",
            "| **Satiety** | The combination of fiber, protein, and a modest amount of healthy fat (if you add nuts/seeds) keeps hunger at bay for 3‑4 hours. |\n",
            "\n",
            "---\n",
            "\n",
            "## 3️⃣ Tweaking the Meal for Specific Goals\n",
            "\n",
            "| Goal | Simple Add‑Ons (≈ 1‑2 Tbsp each) |\n",
            "|------|---------------------------------|\n",
            "| **More Protein** | Greek yogurt, whey or plant‑based protein powder, or a handful of almonds/chia seeds (adds 5‑10 g protein). |\n",
            "| **Lower Carb / Blood‑Sugar Friendly** | Use **½ cup** oats + **½ banana** or swap half the oats for **ground flaxseed** or **unsweetened almond milk**. |\n",
            "| **Extra Healthy Fats** | Add a drizzle of **nut butter** (almond, peanut) or a sprinkle of **walnuts** (≈ 4 g omega‑3). |\n",
            "| **Antioxidant Boost** | Toss in **berries** (blueberries, raspberries) or a pinch of **cinnamon** (may improve insulin sensitivity). |\n",
            "| **Vitamin D & Calcium** | Stir in **fortified plant milk** or a splash of **cow’s milk**; top with **diced cheese** (if you enjoy a savory twist). |\n",
            "| **Weight‑Management** | Keep portion to ½ cup oats, use a smaller banana, and add **½ Tbsp chia** for extra bulk without many extra calories. |\n",
            "\n",
            "---\n",
            "\n",
            "## 4️⃣ Quick “Power‑Breakfast” Recipe (Serves 1)\n",
            "\n",
            "1. **Base**: ½ cup rolled oats + 1 cup water (or ¾ cup water + ¼ cup milk).  \n",
            "2. **Cook**: Bring to boil, simmer 5 min, cover & let sit 2 min.  \n",
            "3. **Mix‑In**:  \n",
            "   - ½ – 1 sliced banana (choose ripe for sweetness).  \n",
            "   - 1 Tbsp chia or ground flaxseed (omega‑3 & extra fiber).  \n",
            "   - 1 Tbsp almond butter (protein & healthy fat).  \n",
            "   - Sprinkle of cinnamon or a dash of vanilla extract (flavor, no extra calories).  \n",
            "4. **Optional Toppings**: A few berries, a handful of toasted nuts, or a drizzle of honey/maple syrup (keep ≤ 1 tsp if watching sugar).  \n",
            "\n",
            "**Total**: ~350 kcal, 12 g protein, 9 g fiber, 12 g healthy fat.\n",
            "\n",
            "---\n",
            "\n",
            "## 5️⃣ Practical Tips\n",
            "\n",
            "| Tip | Why It Helps |\n",
            "|-----|--------------|\n",
            "| **Measure the oats** | Prevents hidden calories; ½ cup dry = ~150 kcal. |\n",
            "| **Use ripe but not over‑ripe bananas** | Gives natural sweetness without excess sugar. |\n",
            "| **Add a pinch of salt** | Enhances flavor and balances the natural sweetness. |\n",
            "| **Prep the night before** | Overnight oats (same ingredients, just soak in liquid) save time and keep the texture creamy. |\n",
            "| **Watch added sweeteners** | A teaspoon of honey adds ~20 kcal; easy to overshoot if you’re counting carbs. |\n",
            "\n",
            "---\n",
            "\n",
            "## 6️⃣ Quick Self‑Check\n",
            "\n",
            "| Question | Your Answer |\n",
            "|----------|-------------|\n",
            "| **What’s your primary goal?** (energy, weight loss, muscle gain, blood‑sugar control, etc.) |  |\n",
            "| **Do you have any dietary restrictions or allergies?** (e.g., gluten, nuts) |  |\n",
            "| **How much time do you have in the morning?** (quick grab‑and‑go vs. sit‑down) |  |\n",
            "\n",
            "Feel free to share your answers, and I can fine‑tune the suggestions (e.g., gluten‑free oats, dairy‑free protein, or a faster “no‑cook” version).\n",
            "\n",
            "Enjoy your oatmeal‑banana breakfast, Chandru! 🌞🥣🍌\n",
            "\n",
            "--- Turn 2 (Testing Memory) ---\n",
            "User: Can you tell me what fruits I added to my breakfast, and mention my name?\n",
            "\n",
            "Agent: Sure thing, **Chandru**!  \n",
            "\n",
            "For your breakfast you mentioned adding **banana** (the fruit) to your oatmeal. 🍌🥣\n",
            "\n"
          ]
        }
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "sample_meal = \"Say my name\"\n",
        "\n",
        "if agent is not None:\n",
        "    print(\"--- Turn 1 ---\")\n",
        "    analysis1 = agent.analyze_meal(sample_meal)\n",
        "    print(f\"User: {sample_meal}\\n\")\n",
        "    print(f\"Agent: {analysis1}\\n\")\n",
        "\n",
        "    # Follow-up turn to verify if it remembers the user's name or previous meal\n",
        "    follow_up = \"Can you tell me what fruits I added to my breakfast, and mention my name?\"\n",
        "    print(\"--- Turn 2 (Testing Memory) ---\")\n",
        "    analysis2 = agent.analyze_meal(follow_up)\n",
        "    print(f\"User: {follow_up}\\n\")\n",
        "    print(f\"Agent: {analysis2}\\n\")\n",
        "else:\n",
        "    print(\"Cannot test. The Nutrition Agent has not been successfully initialized because the Groq client is missing.\")"
      ],
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "id": "_qYWGs5HMFR4",
        "outputId": "11c3b99a-f1a6-46d7-a9b0-707f8931e1ba"
      },
      "execution_count": 25,
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            "--- Turn 1 ---\n",
            "User: Say my name\n",
            "\n",
            "Agent: Chandru.\n",
            "\n",
            "--- Turn 2 (Testing Memory) ---\n",
            "User: Can you tell me what fruits I added to my breakfast, and mention my name?\n",
            "\n",
            "Agent: Sure thing, **Chandru**!  \n",
            "\n",
            "For your breakfast you said you’ll be adding **banana** (the fruit) to your oatmeal. 🍌🥣\n",
            "\n"
          ]
        }
      ]
    },
    {
      "cell_type": "code",
      "source": [],
      "metadata": {
        "id": "lTtISPawN4ec"
      },
      "execution_count": 25,
      "outputs": []
    }
  ]
}

{'nbformat': 4,
 'nbformat_minor': 0,
 'metadata': {'colab': {'provenance': []},
  'kernelspec': {'name': 'python3', 'display_name': 'Python 3'},
  'language_info': {'name': 'python'}},
 'cells': [{'cell_type': 'code',
   'execution_count': 21,
   'metadata': {'id': 'wJr6c-h6LXMR'},
   'outputs': [],
   'source': ['!pip install groq -q']},
  {'cell_type': 'code',
   'source': ['import os\n',
    'from google.colab import userdata\n',
    'from groq import Groq\n',
    '\n',
    '# Retrieve Groq API key from Colab secrets\n',
    'try:\n',
    "    GROQ_API_KEY = userdata.get('GROQ_API_KEY')\n",
    'except Exception:\n',
    '    print("Please add \'GROQ_API_KEY\' to your Colab secrets (the key icon on the left) before continuing.")\n',
    '    GROQ_API_KEY = None\n',
    '\n',
    'if GROQ_API_KEY:\n',
    '    client = Groq(api_key=GROQ_API_KEY)\n',
    '    print("Groq client successfully initialized!")'],
   'metadata': {'colab': {'base_uri': 'https://localhost:8080/'},
    'id': 